# ActionShap — fixed end-to-end experiment notebook

This notebook is the single execution interface for the corrected review response. It downloads/proves data provenance, runs the frozen two-dataset/two-model/five-seed suite, generates component result tables and figures, validates the manuscript, and performs reviewer-specific assertions.

**Important:** the final run is not a smoke test. It refuses to treat missing datasets, failed gates, missing users, or legacy schema-v1 outputs as evidence. Budget 1 and 3 are joint-action sensitivities only; they are not Actionability-Gap conditions. LOO is a deletion oracle, not a positive-gap competitor.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

CODE = Path.cwd()
if CODE.name != "code":
    CODE = Path("paper-ideas/ActionShap/code").resolve()
assert (CODE / "configs/final.yaml").exists(), CODE
os.chdir(CODE)
print("Running from", CODE)


## 1. Environment and deterministic contract

The lock file is the reproducibility contract. The suite uses independent candidate, user, tie, model, and attribution seeds.

In [ ]:
import yaml
config = yaml.safe_load((CODE / "configs/final.yaml").read_text())
assert config["seeds"] == [42,43,44,45,46]
assert config["budget"] == 2
assert config["utility"] == "target_margin"
assert config["n_max"] == 20
assert config["models"] == ["itemknn", "profile"]
print(json.dumps({k: config[k] for k in ("datasets","models","seeds","n_max","budget","utility")}, indent=2))


## 2. Data download and provenance

Review the dataset terms before setting `ACTIONSHAP_ACCEPT_TERMS=1`. The cell deliberately refuses silent downloads. Existing verified files are reused.

In [ ]:
import os
ml = CODE / "data/ml-1m/ratings.dat"
amz = CODE / "data/amazon-digital-music/interactions.csv"
if not (ml.exists() and amz.exists()):
    if os.environ.get("ACTIONSHAP_ACCEPT_TERMS") != "1":
        raise RuntimeError("Missing final datasets. Set ACTIONSHAP_ACCEPT_TERMS=1 after reviewing source terms, then rerun this cell.")
    subprocess.run([sys.executable, "scripts/download_datasets.py", "--dataset", "all", "--accept-dataset-terms"], check=True)
assert ml.exists() and ml.stat().st_size > 0
assert amz.exists() and amz.stat().st_size > 0
print("datasets ready", ml.stat().st_size, amz.stat().st_size)


## 3. Frozen final suite

This executes convergence first, then the primary matrix, full-catalogue robustness, and predeclared sensitivities. Do not edit the YAML after this cell begins.

In [ ]:
subprocess.run([sys.executable, "scripts/run_final_suite.py", "--config", "configs/final.yaml"], check=True)


## 4. Generate corrected paper assets

The generator now writes:
- `aia_components.tex`: deletion AIA, bounded AIA, and gap for all five methods;
- `intervention_outcomes.tex`: effect, harm/success, abstention, and conditional regret;
- component figure with three panels;
- no budget rows in gap assets;
- no Shapley–LOO gap headline comparison.

In [ ]:
subprocess.run([sys.executable, "scripts/make_paper_assets.py", "--raw", "results/schema-v2", "--out", "../paper"], check=True)
subprocess.run([sys.executable, "scripts/validate_manuscript.py", "--require-final"], check=True)


## 5. Reviewer-specific scientific assertions

In [ ]:
import pandas as pd
final = CODE.parent / "paper/final"
gap = pd.read_csv(final / "tables/actionability_gap_robustness.csv")
components = pd.read_csv(final / "tables/aia_components.csv")
assert set(gap.method) == {"shapley_mc","lime","loo","greedy_cf","random"}
assert not gap.condition_label.str.contains("B=1|B=3|budget", case=False, regex=True).any()
assert set(components.component) == {"Deletion AIA","Bounded AIA","Gap (bounded - deletion)"}
# Algebraic identity check at the method/condition summary level wherever finite.
for _, row in components.loc[components.component == "Gap (bounded - deletion)"].iterrows():
    sel = components[(components.dataset == row.dataset) & (components.model == row.model) & (components.evaluation_mode == row.evaluation_mode) & (components.condition == row.condition) & (components.method == row.method)]
    if len(sel) == 3:
        vals = dict(zip(sel.component, sel['mean']))
        assert abs(vals['Gap (bounded - deletion)'] - (vals['Bounded AIA'] - vals['Deletion AIA'])) < 1e-8
text = (CODE.parent / "paper/paper.tex").read_text()
for forbidden in ["only method with a positive", "22 comparisons", "uniquely intervention-robust", "Shapley alone improves"]:
    assert forbidden.lower() not in text.lower(), forbidden
assert "aia_components.tex" in text and "intervention_outcomes.tex" in text
print("review assertions passed", len(gap), "gap rows", len(components), "component rows")


## 6. Release checklist

A submission is allowed only if every item passes.

In [ ]:
subprocess.run([sys.executable, "scripts/validate_review_contract.py", "--paper-root", "../paper"], check=True)


In [ ]:
report = json.loads((final / "manifests/validation_report.json").read_text())
assert report["status"] == "PASS", report
assert report["errors"] == []
print(json.dumps(report, indent=2))
print("READY: schema-v2 PASS, manuscript PASS, reviewer contradiction checks PASS")
